# ML-03 — Frame Your Lane as an ML Task

**Intern:** Sathwik | **Track:** Machine Learning | **Assignment:** ML-03
**Lane:** Lane 2: Refresh / Content Opportunity Scoring

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type:** **Ranking / Scoring** (specifically, learning-to-rank or scoring a priority queue).

**Why?**
The core business decision here is: **"Which content pages should the SEO/editorial team review first for a refresh, expansion, or protection?"**
1. **Resource Constraints:** Reviewing articles takes significant human effort (reading, editing, re-publishing). A reviewer can only look at a small number of pages per week (e.g., 20 or 50 pages).
2. **Value of Order:** It is not enough to classify pages as simply "declining" or "not declining" (classification). We need to prioritize pages by the *potential impact* or *severity of decline*. The top-ranked pages should have the highest expected return on investment (ROI) or represent the highest traffic risk.
3. **Task Definition:** Therefore, we assign a numeric "opportunity score" (between 0 and 1 or unbounded priority) to each page and sort the pages in descending order. This makes it a **ranking and scoring** task.

In [1]:
# Showing the sorting/ranking nature of our goal
import pandas as pd
import numpy as np

print("Task Type: Ranking / Scoring")
print("Goal: Rank N content items to produce a prioritized queue of size K (e.g., K=50).")

Task Type: Ranking / Scoring
Goal: Rank N content items to produce a prioritized queue of size K (e.g., K=50).


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**What we predict:** The target is `is_declining_label`, which is a binary proxy representing whether a page is in a state of organic search traffic decline.
- In a production environment, this is a proxy target because the true label we want to optimize is "increase in traffic after a refresh". However, "refresh opportunity" is not directly observable before the action is taken.
- Therefore, we use a proxy target: **organic search performance decline**. We want to identify pages that are currently losing traction so the team can refresh them to reverse the decline.

**Where the label comes from:**
- The label is derived from an **observed outcome**: the performance trend over time.
- In this starter dataset, `is_declining_label` is a binary flag (1 for declining, 0 for stable/growing) derived from the observed GSC performance metrics: specifically, when the search click/impression trend direction (`trend_direction`) is measured as `"down"`.
- *Caution/Leakage:* Because the label is calculated from `trend_pct` and `trend_direction`, these trend features must never be used as inputs to the model.

In [2]:
# Load dataset to inspect the target distribution
import os
path = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(path):
    path = "../../" + path
df = pd.read_csv(path)

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

declining_count = df["is_declining_label"].sum()
total_count = len(df)
declining_rate = df["is_declining_label"].mean()

print(f"Total pages: {total_count}")
print(f"Declining pages: {declining_count} ({declining_rate * 100:.1f}%)")
print(f"Target Column Name: 'is_declining_label' (binary, observed outcome-based proxy)")

Total pages: 30000
Declining pages: 16262 (54.2%)
Target Column Name: 'is_declining_label' (binary, observed outcome-based proxy)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success Metric:** **Precision@K** (specifically **Precision@50** and **Precision@20**).

**Why we defend this metric:**
- The end-user (SEO/editor team) has a fixed queue capacity. If they review the top 50 pages, we want to maximize the proportion of those pages that genuinely needed a refresh (true positives).
- If Precision@50 is low (e.g., 20%), 80% of the editor's time is wasted reviewing healthy pages, causing high friction and loss of trust in the system.
- General metrics like ROC-AUC or average accuracy are less representative because they evaluate the entire dataset of 30,000 pages. Since the reviewer will never read page number 10,000, we only care about the quality at the very top of the ranked list.

**What number means 'good'?**
- A simple hand-written heuristic rule (e.g., ranking purely by stale x visible impressions) gets a **Precision@50 of 0.240**.
- The baseline Random Forest model achieves a **Precision@50 of 0.740** on the validation set.
- Therefore, we define:
  - **Baseline:** Precision@50 = 0.240 (hand-written rule)
  - **Good:** Precision@50 **>= 0.700** on holdout clients.
  - **Excellent:** Precision@50 **>= 0.750**.

In [3]:
# Define Precision@K function to show how it's measured
def precision_at_k(y_true, scores, k=50):
    # Sort by score descending and get top K labels
    top_indices = np.argsort(scores)[::-1][:k]
    top_labels = y_true.iloc[top_indices]
    return top_labels.mean()

# Let's test it on a dummy random ranker
np.random.seed(42)
dummy_scores = np.random.rand(len(df))
dummy_p50 = precision_at_k(df["is_declining_label"], dummy_scores, k=50)
print(f"Dummy Random Ranker Precision@50: {dummy_p50:.3f} (expected near random rate of {declining_rate:.3f})")

Dummy Random Ranker Precision@50: 0.560 (expected near random rate of 0.542)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis:** **One row = One pseudonymous content item (page) for a specific client**, containing search volume, GSC impressions/clicks, GA4 traffic, and page characteristics aggregated over a trailing 90-day window.

Below we load the starter dataset, verify its shape, and display a few key features that represent the unit of analysis.

In [4]:
# Show the unit of analysis
import pandas as pd
import os

path = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(path):
    path = "../../" + path
df = pd.read_csv(path)

# Select a subset of representative columns for display
display_cols = [
    "content_id", 
    "client_id", 
    "content_type", 
    "impressions_90d", 
    "ctr", 
    "avg_position", 
    "days_since_last_update", 
    "is_declining_label"
]
# Populate target label
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Starter Dataframe Shape: {df.shape[0]} rows (pages) x {df.shape[1]} columns")
print("\nRepresentative slice of the dataframe (first 5 rows):")
df[display_cols].head()

Starter Dataframe Shape: 30000 rows (pages) x 45 columns

Representative slice of the dataframe (first 5 rows):


,content_id,client_id,content_type,impressions_90d,ctr,avg_position,days_since_last_update,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,0.76,10.6,20,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,0.05,20.3,25,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,0.09,36.5,20,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,0.49,6.2,22,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,0.13,44.0,14,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed hand-written rule (like "review pages if `days_since_last_update` > 180 and `impressions` > 500") struggles because:
1. **Multi-dimensional interactions:** Traffic decline is not caused by a single factor. A page might decline because its position dropped slightly, because click-through rates fell, or because its category is losing interest. An if-statement cannot easily weigh and balance 10+ numerical features (impressions, clicks, word count, cpc, avg_position, scroll rate, engagement rate) simultaneously.
2. **Non-linear thresholds:** The relationship between features and decline is non-linear. For example, a decline in rank from position 2 to 5 is catastrophic for click-through rate, whereas a shift from position 22 to 25 is negligible. A simple rule cannot easily model these position-dependent curves.
3. **Client/Category heterogeneity:** Different clients and content types have different baselines. A "stale" page for a news-heavy client (e.g., 30 days old) might be extremely urgent, while a "stale" page for evergreen content (e.g., 360 days old) might be perfectly fine. ML (especially tree-based models like Random Forests) can learn these complex splits and interactions automatically.

In [5]:
# Showing the correlation of multiple features with the target to demonstrate complexity
features_to_check = [
    "impressions_90d", 
    "ctr", 
    "avg_position", 
    "days_since_last_update", 
    "word_count", 
    "scroll_rate", 
    "engagement_rate"
]

# Compute correlation matrix with the target
correlations = df[features_to_check].corrwith(df["is_declining_label"])
print("Correlations of observable features with 'is_declining_label':")
for feat, corr in correlations.items():
    print(f"  {feat:25s} : {corr:+.4f}")

# Show that no single feature is highly correlated, proving the need for multi-feature modeling
print("\nNo single feature has a strong linear relationship (corr > 0.4 or < -0.4).")
print("This indicates that the signal is weak/complex individually and requires non-linear combinations (ML) to be predicted accurately.")

Correlations of observable features with 'is_declining_label':
  impressions_90d           : -0.0182
  ctr                       : -0.0619
  avg_position              : -0.0290
  days_since_last_update    : +0.0814
  word_count                : +0.0902
  scroll_rate               : -0.0030
  engagement_rate           : -0.0127

No single feature has a strong linear relationship (corr > 0.4 or < -0.4).
This indicates that the signal is weak/complex individually and requires non-linear combinations (ML) to be predicted accurately.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.